In [2]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("device name:", torch.cuda.get_device_name(0))
print("capability:", torch.cuda.get_device_capability(0))

torch version: 2.8.0+cu128
cuda available: True
cuda version: 12.8
device name: NVIDIA A100-SXM4-80GB
capability: (8, 0)


In [3]:
%pip install -q --upgrade pip setuptools wheel

%pip install -q --upgrade \
  "transformers>=4.57.0,<5.0.0" \
  "accelerate>=1.1.0,<2.0.0" \
  "datasets>=3.0.1,<4.0.0" \
  "evaluate>=0.4.3,<1.0.0" \
  "trl>=0.24.0,<1.0.0" \
  "peft>=0.18.0,<1.0.0" \
  "qwen-vl-utils>=0.0.14" \
  "Pillow>=9.4.0" \
  "scikit-learn>=1.3.0" \
  "tensorboard" \
  "wandb" \
  "rich"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -q hf_transfer

Note: you may need to restart the kernel to use updated packages.


Kernel > Restart Kernel

In [5]:
import torch
import transformers
import accelerate
import datasets
import trl
import peft
import qwen_vl_utils

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0))
print("capability:", torch.cuda.get_device_capability(0))

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)

torch: 2.8.0+cu128
cuda: 12.8
gpu: NVIDIA A100-SXM4-80GB
capability: (8, 0)
transformers: 4.57.6
accelerate: 1.14.0
datasets: 3.6.0
trl: 0.29.1
peft: 0.19.1


In [6]:
import io
import json
import os
import random

import numpy as np
import torch
import wandb

from PIL import Image
from datasets import load_dataset
from sklearn.model_selection import train_test_split

from transformers import AutoProcessor, AutoModelForImageTextToText
from trl import SFTConfig, SFTTrainer
from qwen_vl_utils import process_vision_info
from peft import LoraConfig

## 1. 기본 설정

In [7]:
wandb.init(mode="disabled")

SEED = 42
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
OUTPUT_DIR = "output_dir_qwen3_vl_4b_instruct_lora"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

torch: 2.8.0+cu128
cuda: 12.8
cuda available: True
gpu: NVIDIA A100-SXM4-80GB
capability: (8, 0)


## 2. 프롬프트 정의

In [8]:
system_message = "당신은 이미지와 제품명(name)으로부터 패션/스타일 정보를 추론하는 분류 모델입니다."

prompt = """입력 정보:
- name: {name}
- image: [image]

위 정보를 바탕으로, 아래 7가지 key에 대한 값을 JSON 형태로 추론해 주세요:
1) gender
2) masterCategory
3) subCategory
4) season
5) usage
6) baseColour
7) articleType

출력 시 **아래 JSON 예시 형태**를 반드시 지키세요:
{{
  "gender": "예시값",
  "masterCategory": "예시값",
  "subCategory": "예시값",
  "season": "예시값",
  "usage": "예시값",
  "baseColour": "예시값",
  "articleType": "예시값"
}}

# 예시
{{
  "gender": "Men",
  "masterCategory": "Accessories",
  "subCategory": "Eyewear",
  "season": "Winter",
  "usage": "Casual",
  "baseColour": "Blue",
  "articleType": "Sunglasses"
}}

# 주의
- 7개 항목 이외의 정보(텍스트, 문장 등)는 절대 포함하지 마세요.
"""

## 3. 데이터셋 로드 및 라벨 설정

In [9]:
def combine_cols_to_label(example):
    label_dict = {
        "gender": example["gender"],
        "masterCategory": example["masterCategory"],
        "subCategory": example["subCategory"],
        "season": example["season"],
        "usage": example["usage"],
        "baseColour": example["baseColour"],
        "articleType": example["articleType"],
    }

    example["label"] = json.dumps(label_dict, ensure_ascii=False)
    return example

In [10]:
def format_data(sample):
    buffer = io.BytesIO()
    sample["image"].save(buffer, format="PNG")
    buffer.seek(0)

    image = Image.open(buffer).convert("RGB")

    return {
        "messages": [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": system_message,
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt.format(name=sample["productDisplayName"]),
                    },
                    {
                        "type": "image",
                        "image": image,
                    },
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": sample["label"],
                    }
                ],
            },
        ],
    }

In [11]:
dataset = load_dataset("ashraq/fashion-product-images-small", split="train")
dataset_add_label = dataset.map(combine_cols_to_label)
dataset_add_label = dataset_add_label.shuffle(seed=4242)

formatted_dataset = [format_data(row) for row in dataset_add_label]

train_dataset, test_dataset = train_test_split(
    formatted_dataset,
    test_size=0.9,
    random_state=42,
)

README.md:   0%|          | 0.00/867 [00:00<?, ?B/s]

data/train-00000-of-00002-6cff4c59f91661(…):   0%|          | 0.00/136M [00:00<?, ?B/s]

data/train-00001-of-00002-bb459e5ac5f01e(…):   0%|          | 0.00/135M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44072 [00:00<?, ? examples/s]

Map:   0%|          | 0/44072 [00:00<?, ? examples/s]

In [12]:
print("학습 데이터의 개수:", len(train_dataset))
print("테스트 데이터의 개수:", len(test_dataset))
print("샘플 데이터:")
print(train_dataset[0])

학습 데이터의 개수: 4407
테스트 데이터의 개수: 39665
샘플 데이터:
{'messages': [{'role': 'system', 'content': [{'type': 'text', 'text': '당신은 이미지와 제품명(name)으로부터 패션/스타일 정보를 추론하는 분류 모델입니다.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '입력 정보:\n- name: Mr.Men Men\'s Charcoal White T-shirt\n- image: [image]\n\n위 정보를 바탕으로, 아래 7가지 key에 대한 값을 JSON 형태로 추론해 주세요:\n1) gender\n2) masterCategory\n3) subCategory\n4) season\n5) usage\n6) baseColour\n7) articleType\n\n출력 시 **아래 JSON 예시 형태**를 반드시 지키세요:\n{\n  "gender": "예시값",\n  "masterCategory": "예시값",\n  "subCategory": "예시값",\n  "season": "예시값",\n  "usage": "예시값",\n  "baseColour": "예시값",\n  "articleType": "예시값"\n}\n\n# 예시\n{\n  "gender": "Men",\n  "masterCategory": "Accessories",\n  "subCategory": "Eyewear",\n  "season": "Winter",\n  "usage": "Casual",\n  "baseColour": "Blue",\n  "articleType": "Sunglasses"\n}\n\n# 주의\n- 7개 항목 이외의 정보(텍스트, 문장 등)는 절대 포함하지 마세요.\n'}, {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=60x80 at 0x75F08CC603B0>}]}, {'role

## 4. 프로세서 및 모델 로드

In [13]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)

processor.tokenizer.padding_side = "right"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model.config.use_cache = False

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

## 5. 채팅 템플릿 확인

In [14]:
text = processor.apply_chat_template(
    train_dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)

print("채팅 템플릿 적용 결과:")
print(text)

채팅 템플릿 적용 결과:
<|im_start|>system
당신은 이미지와 제품명(name)으로부터 패션/스타일 정보를 추론하는 분류 모델입니다.<|im_end|>
<|im_start|>user
입력 정보:
- name: Mr.Men Men's Charcoal White T-shirt
- image: [image]

위 정보를 바탕으로, 아래 7가지 key에 대한 값을 JSON 형태로 추론해 주세요:
1) gender
2) masterCategory
3) subCategory
4) season
5) usage
6) baseColour
7) articleType

출력 시 **아래 JSON 예시 형태**를 반드시 지키세요:
{
  "gender": "예시값",
  "masterCategory": "예시값",
  "subCategory": "예시값",
  "season": "예시값",
  "usage": "예시값",
  "baseColour": "예시값",
  "articleType": "예시값"
}

# 예시
{
  "gender": "Men",
  "masterCategory": "Accessories",
  "subCategory": "Eyewear",
  "season": "Winter",
  "usage": "Casual",
  "baseColour": "Blue",
  "articleType": "Sunglasses"
}

# 주의
- 7개 항목 이외의 정보(텍스트, 문장 등)는 절대 포함하지 마세요.
<|vision_start|><|image_pad|><|vision_end|><|im_end|>
<|im_start|>assistant
{"gender": "Men", "masterCategory": "Apparel", "subCategory": "Topwear", "season": "Fall", "usage": "Casual", "baseColour": "Grey", "articleType": "Tshirts"}<|im_end|>



## 6. 로라

In [15]:
peft_config = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.05,
    r=64,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type="CAUSAL_LM",
)

## 7. Collate_fn 설정

In [16]:
def get_special_token_ids(tokenizer):
    special_tokens = [
        "<|vision_start|>",
        "<|vision_end|>",
        "<|image_pad|>",
        "<|video_pad|>",
    ]

    token_ids = []

    for token in special_tokens:
        token_id = tokenizer.convert_tokens_to_ids(token)

        if token_id is None:
            continue

        if tokenizer.unk_token_id is not None and token_id == tokenizer.unk_token_id:
            continue

        token_ids.append(token_id)

    return list(set(token_ids))

In [17]:
def find_subsequence(sequence, subsequence):
    if len(subsequence) == 0:
        return -1

    last_match = -1

    for i in range(len(sequence) - len(subsequence) + 1):
        if sequence[i:i + len(subsequence)] == subsequence:
            last_match = i

    return last_match

In [18]:
def mask_non_assistant_tokens(labels, input_ids, tokenizer):
    assistant_prefix = "<|im_start|>assistant\n"
    assistant_prefix_ids = tokenizer.encode(
        assistant_prefix,
        add_special_tokens=False,
    )

    for row_idx in range(input_ids.shape[0]):
        row_input_ids = input_ids[row_idx].tolist()

        start_idx = find_subsequence(
            row_input_ids,
            assistant_prefix_ids,
        )

        if start_idx == -1:
            labels[row_idx, :] = -100
            continue

        answer_start_idx = start_idx + len(assistant_prefix_ids)
        labels[row_idx, :answer_start_idx] = -100

    return labels

In [19]:
def collate_fn(examples):
    texts = [
        processor.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        for example in examples
    ]

    image_inputs = []
    video_inputs = []

    for example in examples:
        image_input, video_input = process_vision_info(example["messages"])

        if image_input is not None:
            image_inputs.extend(image_input)

        if video_input is not None and len(video_input) > 0:
            video_inputs.extend(video_input)

    processor_kwargs = {
        "text": texts,
        "return_tensors": "pt",
        "padding": True,
    }

    if len(image_inputs) > 0:
        processor_kwargs["images"] = image_inputs

    if len(video_inputs) > 0:
        processor_kwargs["videos"] = video_inputs

    batch = processor(**processor_kwargs)

    labels = batch["input_ids"].clone()

    if processor.tokenizer.pad_token_id is not None:
        labels[labels == processor.tokenizer.pad_token_id] = -100

    for special_token_id in get_special_token_ids(processor.tokenizer):
        labels[labels == special_token_id] = -100

    labels = mask_non_assistant_tokens(
        labels=labels,
        input_ids=batch["input_ids"],
        tokenizer=processor.tokenizer,
    )

    batch["labels"] = labels

    return batch

## 8. Collate 함수 테스트

In [20]:
# 단일 예시 확인
example = train_dataset[0]  # 데이터셋의 첫 번째 아이템

print("단일 예시 데이터:")
print(example)

# collate_fn 테스트 (배치 크기 1로)
batch = collate_fn([example])

print("\n처리된 배치 데이터:")
print("입력 ID 형태:", batch["input_ids"].shape)
print("어텐션 마스크 형태:", batch["attention_mask"].shape)

if "pixel_values" in batch:
    print("이미지 픽셀 형태:", batch["pixel_values"].shape)

if "image_grid_thw" in batch:
    print("이미지 grid 형태:", batch["image_grid_thw"].shape)
    print("이미지 grid 값:")
    print(batch["image_grid_thw"])

if "video_grid_thw" in batch:
    print("비디오 grid 형태:", batch["video_grid_thw"].shape)
    print("비디오 grid 값:")
    print(batch["video_grid_thw"])

단일 예시 데이터:
{'messages': [{'role': 'system', 'content': [{'type': 'text', 'text': '당신은 이미지와 제품명(name)으로부터 패션/스타일 정보를 추론하는 분류 모델입니다.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '입력 정보:\n- name: Mr.Men Men\'s Charcoal White T-shirt\n- image: [image]\n\n위 정보를 바탕으로, 아래 7가지 key에 대한 값을 JSON 형태로 추론해 주세요:\n1) gender\n2) masterCategory\n3) subCategory\n4) season\n5) usage\n6) baseColour\n7) articleType\n\n출력 시 **아래 JSON 예시 형태**를 반드시 지키세요:\n{\n  "gender": "예시값",\n  "masterCategory": "예시값",\n  "subCategory": "예시값",\n  "season": "예시값",\n  "usage": "예시값",\n  "baseColour": "예시값",\n  "articleType": "예시값"\n}\n\n# 예시\n{\n  "gender": "Men",\n  "masterCategory": "Accessories",\n  "subCategory": "Eyewear",\n  "season": "Winter",\n  "usage": "Casual",\n  "baseColour": "Blue",\n  "articleType": "Sunglasses"\n}\n\n# 주의\n- 7개 항목 이외의 정보(텍스트, 문장 등)는 절대 포함하지 마세요.\n'}, {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=60x80 at 0x75F08CC603B0>}]}, {'role': 'assistant', 'content': [{'typ

In [21]:
print("레이블 형태:", batch["labels"].shape)


# 입력 ID 전체 확인
print("\n입력에 대한 정수 인코딩 결과:")
print(batch["input_ids"][0])


# 레이블 전체 확인
print("\n레이블에 대한 정수 인코딩 결과:")
print(batch["labels"][0])

레이블 형태: torch.Size([1, 575])

입력에 대한 정수 인코딩 결과:
tensor([151644,   8948,    198,  64795,  82528,  33704,  90667,  21329,  80573,
        138017,  79632,   3153,      8,  42039, 126558,  45104,    101,  92031,
            14, 141274,  32077,  60039,  18411,  57835, 126605,  42905, 128618,
         97929,  54070, 142713,  78952,     13, 151645,    198, 151644,    872,
           198,  43866,  28754,  60039,    510,     12,    829,     25,   4392,
          1321,    268,  11012,    594,   4864,  40465,   5807,    350,  33668,
           198,     12,   2168,     25,    508,   1805,   2533,  80901,  60039,
         18411,  81718, 144059,  42039,     11, 136646,    220,     22,  19969,
         21329,   1376,  19391, 128605,  93668,   4718, 141966,  17380,  57835,
        126605,  33883,  55673,  50302,    510,     16,      8,   9825,    198,
            17,      8,   7341,   6746,    198,     18,      8,   1186,   6746,
           198,     19,      8,   3200,    198,     20,      8,  10431, 

In [22]:
# 토큰 디코딩 예시
decoded_text = processor.tokenizer.decode(batch["input_ids"][0])

print("\n디코딩된 텍스트:")
print(decoded_text)


# 실제 loss 계산 대상 토큰만 확인
valid_label_count = (batch["labels"][0] != -100).sum().item()

print("\nloss 계산 대상 토큰 수:")
print(valid_label_count)

label_ids = batch["labels"][0]
label_ids_for_decode = label_ids[label_ids != -100]

decoded_labels = processor.tokenizer.decode(label_ids_for_decode)

print("\nloss 계산 대상 labels 디코딩 결과:")
print(decoded_labels)


디코딩된 텍스트:
<|im_start|>system
당신은 이미지와 제품명(name)으로부터 패션/스타일 정보를 추론하는 분류 모델입니다.<|im_end|>
<|im_start|>user
입력 정보:
- name: Mr.Men Men's Charcoal White T-shirt
- image: [image]

위 정보를 바탕으로, 아래 7가지 key에 대한 값을 JSON 형태로 추론해 주세요:
1) gender
2) masterCategory
3) subCategory
4) season
5) usage
6) baseColour
7) articleType

출력 시 **아래 JSON 예시 형태**를 반드시 지키세요:
{
  "gender": "예시값",
  "masterCategory": "예시값",
  "subCategory": "예시값",
  "season": "예시값",
  "usage": "예시값",
  "baseColour": "예시값",
  "articleType": "예시값"
}

# 예시
{
  "gender": "Men",
  "masterCategory": "Accessories",
  "subCategory": "Eyewear",
  "season": "Winter",
  "usage": "Casual",
  "baseColour": "Blue",
  "articleType": "Sunglasses"
}

# 주의
- 7개 항목 이외의 정보(텍스트, 문장 등)는 절대 포함하지 마세요.
<|vision_start|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_p

## 9. 학습 설정

In [23]:
args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    gradient_checkpointing=True,
    optim="adamw_torch_fused",

    logging_steps=10,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,

    bf16=True,
    fp16=False,

    learning_rate=1e-4,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",

    push_to_hub=False,
    remove_unused_columns=False,

    dataset_kwargs={
        "skip_prepare_dataset": True,
    },

    report_to=None,
)

## 10. Trainer 생성

In [24]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
    processing_class=processor,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.303800
20,0.080000
30,0.057800
40,0.049500
50,0.045200
60,0.034900
70,0.035500
80,0.043100
90,0.042800
100,0.033200


TrainOutput(global_step=552, training_loss=0.036697893808393375, metrics={'train_runtime': 7581.41, 'train_samples_per_second': 1.163, 'train_steps_per_second': 0.073, 'total_flos': 1.2705585830232576e+17, 'train_loss': 0.036697893808393375})

In [25]:
from peft import PeftModel
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

In [26]:
adapter_path = "./output_dir_qwen3_vl_4b_instruct_lora/checkpoint-552"
base_model_id = "Qwen/Qwen3-VL-4B-Instruct"
merged_path = "merged"

In [27]:
# 베이스 모델 로드
model = AutoModelForImageTextToText.from_pretrained(
    base_model_id,
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [28]:
# LoRA 어댑터 로드 및 병합
print(f"Loading and merging PEFT from: {adapter_path}")
peft_model = PeftModel.from_pretrained(model, adapter_path)
merged_model = peft_model.merge_and_unload()

Loading and merging PEFT from: ./output_dir_qwen3_vl_4b_instruct_lora/checkpoint-552


In [29]:
merged_model.save_pretrained(
    merged_path,
    safe_serialization=True,
    max_shard_size="2GB",
)

In [30]:
# 프로세서 로드
processor = AutoProcessor.from_pretrained(base_model_id)
processor.save_pretrained(merged_path)

[]

In [31]:
from huggingface_hub import HfApi

api = HfApi()

In [32]:
username = "iamjoon"
MODEL_NAME = "Qwen3-VL-4B-Instruct-fashion-product-images-small-checkpoint-552"

In [33]:
api.create_repo(
    token="hf_여러분의 키 값",
    repo_id=f"{username}/{MODEL_NAME}",
    repo_type="model",
)

RepoUrl('https://huggingface.co/iamjoon/Qwen3-VL-4B-Instruct-fashion-product-images-small-checkpoint-552', endpoint='https://huggingface.co', repo_type='model', repo_id='iamjoon/Qwen3-VL-4B-Instruct-fashion-product-images-small-checkpoint-552')

In [34]:
api.upload_folder(
    token="hf_여러분의 키 값",
    repo_id=f"{username}/{MODEL_NAME}",
    folder_path="merged",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/iamjoon/Qwen3-VL-4B-Instruct-fashion-product-images-small-checkpoint-552/commit/4d5ae1122c1dd67124053b20ebb604f80268aa3c', commit_message='Upload folder using huggingface_hub', commit_description='', oid='4d5ae1122c1dd67124053b20ebb604f80268aa3c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/iamjoon/Qwen3-VL-4B-Instruct-fashion-product-images-small-checkpoint-552', endpoint='https://huggingface.co', repo_type='model', repo_id='iamjoon/Qwen3-VL-4B-Instruct-fashion-product-images-small-checkpoint-552'), pr_revision=None, pr_num=None)